In [1]:
# CELL 1: SETUP & DIRECT GCP BUCKET INGESTION (NO REPO DEPENDENCY)
import os
import io
import json
import time
from pathlib import Path
from collections import defaultdict
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from google.cloud import storage

print("=" * 70)
print("STEP 1: INGESTING HUL_VARIANTS MANIFEST DIRECTLY FROM GCS")
print("=" * 70)

# 1. Direct GCS Paths & Storage Client
DATA_ROOT = "gs://marketshare-datasets/derived_manifests/hul_variants"
RPC_JSON_URI = f"{DATA_ROOT}/rpc.json"
print(f"Reading manifest directly from: {RPC_JSON_URI}")

storage_client = storage.Client()

def read_gcs_bytes(gcs_uri: str) -> bytes:
    """Download raw bytes directly from any gs:// URI."""
    if not gcs_uri.startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got: {gcs_uri}")
    path_without_prefix = gcs_uri[5:]
    bucket_name, blob_name = path_without_prefix.split("/", 1)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    return blob.download_as_bytes()

def resolve_gcs_uri(uri_or_path: str) -> str:
    """Ensure path is a fully qualified gs:// URI."""
    if uri_or_path.startswith("gs://"):
        return uri_or_path
    return f"{DATA_ROOT.rstrip('/')}/{uri_or_path.lstrip('/')}"

# 2. Download and parse rpc.json manifest
raw_json = read_gcs_bytes(RPC_JSON_URI)
manifest = json.loads(raw_json.decode("utf-8"))

classes = manifest["classes"]  # 1,922 Brand__Variant classes
gallery = manifest.get("gallery", {})
splits = manifest.get("splits", {})

# 3. Build sorted contiguous class mappings (0 to 1921)
sorted_sku_ids = sorted([int(k) for k in classes.keys()])
num_classes = len(sorted_sku_ids)
print(f"Total Canonical Brand-Variant Classes: {num_classes}")

sku_to_idx = {sku_id: idx for idx, sku_id in enumerate(sorted_sku_ids)}
idx_to_sku = {idx: sku_id for idx, sku_id in enumerate(sorted_sku_ids)}

# 4. Save deployment metadata mapping for inference/leaderboard scoring
idx_to_meta = {}
for idx, sku_id in idx_to_sku.items():
    meta = classes.get(str(sku_id), {})
    b = meta.get("brand", "Unknown")
    v = meta.get("base_variant", meta.get("variant", "Default"))
    idx_to_meta[idx] = {
        "sku_id": int(sku_id),
        "brand": b,
        "base_variant": v,
        "label": f"{b}__{v}",
        "category": meta.get("category", "General"),
        "is_hul": meta.get("is_hul", True)
    }

with open("variant_classes.json", "w") as f:
    json.dump(idx_to_meta, f, indent=2)
print("✔ Saved 'variant_classes1.json'.")

# 5. Union of gallery + train (guarantees 100% of 1,922 classes)
train_by_uri = {}

# Step 5a: Add gallery reference crops
for cid_str, uris in gallery.items():
    cid = int(cid_str)
    for u in uris:
        full_u = resolve_gcs_uri(u)
        train_by_uri[full_u] = {
            "uri": full_u,
            "sku_id": cid,
            "target_idx": sku_to_idx[cid]
        }

# Step 5b: Merge splits["train"]
for row in splits.get("train", []):
    u = row.get("uri") or row.get("image")
    cid = int(row["products"][0])
    if cid in sku_to_idx:
        full_u = resolve_gcs_uri(u)
        train_by_uri[full_u] = {
            "uri": full_u,
            "sku_id": cid,
            "target_idx": sku_to_idx[cid]
        }

full_train_records = list(train_by_uri.values())
unique_train_classes = len({r["sku_id"] for r in full_train_records})

print(f"\n📊 DATASET COMPILATION SUMMARY:")
print(f"  • Total Union Train Images: {len(full_train_records):,} (Expected ~22,104)")
print(f"  • Total Classes Covered:    {unique_train_classes} / {num_classes} (100.0%)")
print(f"  • Validation Split Count:   {len(splits.get('val', [])):,} images")
print(f"  • Test Split Count:         {len(splits.get('test', [])):,} images")

STEP 1: INGESTING HUL_VARIANTS MANIFEST DIRECTLY FROM GCS
Reading manifest directly from: gs://marketshare-datasets/derived_manifests/hul_variants/rpc.json
Total Canonical Brand-Variant Classes: 1922
✔ Saved 'variant_classes1.json'.

📊 DATASET COMPILATION SUMMARY:
  • Total Union Train Images: 22,104 (Expected ~22,104)
  • Total Classes Covered:    1922 / 1922 (100.0%)
  • Validation Split Count:   1,900 images
  • Test Split Count:         1,905 images


In [2]:
# CELL 2: FAST PYTORCH DATASET (ATOMIC, COLLISION-PROOF & LOSSLESS)
import os
import io
import time
import hashlib
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

CACHE_DIR = Path("cache/crops")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


class FastBrandVariantDataset(Dataset):
    def __init__(self, records, transform=None, is_split_row=False):
        self.records = records
        self.transform = transform
        self.is_split_row = is_split_row

    def __len__(self):
        return len(self.records)

    def _get_local_or_download(self, uri_or_path: str) -> Image.Image:
        # 1. Resolve full GCS URI
        full_uri = resolve_gcs_uri(uri_or_path)

        # 2. Collision-proof filename using SHA-256 hash of complete URI
        uri_hash = hashlib.sha256(full_uri.encode("utf-8")).hexdigest()
        cache_path = CACHE_DIR / f"{uri_hash}.bin"

        # 3. Fast path: load directly from local SSD cache if present
        if cache_path.exists():
            try:
                with open(cache_path, "rb") as f:
                    return Image.open(f).convert("RGB")
            except Exception:
                pass  # Fallback to re-download if cache was corrupted

        # 4. Download from GCS on Epoch 1
        try:
            raw = read_gcs_bytes(full_uri)

            # --- ATOMIC WRITE (Prevents multi-worker write collisions) ---
            # Write to a worker-specific temp file first
            tmp_path = CACHE_DIR / f"{uri_hash}.tmp_{os.getpid()}_{time.time_ns()}"
            tmp_path.write_bytes(raw)
            # Atomically rename to final destination (OS-level safe swap)
            os.replace(tmp_path, cache_path)

            return Image.open(io.BytesIO(raw)).convert("RGB")
        except Exception:
            # Fallback neutral canvas if image is missing/corrupted
            return Image.new("RGB", (224, 224), (240, 240, 240))

    def __getitem__(self, idx):
        item = self.records[idx]

        if self.is_split_row:
            # Validation / test split row
            uri = item.get("uri") or item.get("image")
            cid = int(item["products"][0])
            target_idx = sku_to_idx[cid]
            boxes = item.get("boxes", [])
            box = boxes[0] if boxes else None
        else:
            # Train union records
            uri = item["uri"]
            target_idx = item["target_idx"]
            box = None

        img = self._get_local_or_download(uri)

        # Crop bounding box if coordinates exist
        if box:
            x1, y1, x2, y2 = box
            w, h = img.size
            if x2 > x1 and y2 > y1 and (x2 - x1 < w or y2 - y1 < h):
                img = img.crop((x1, y1, x2, y2))

        if self.transform:
            img = self.transform(img)

        return img, target_idx


# Standard transforms for MaxViT (224x224)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Preparing train and val datasets...")
train_dataset = FastBrandVariantDataset(full_train_records, transform=train_transforms)
val_dataset = FastBrandVariantDataset(splits.get("val", []), transform=val_transforms, is_split_row=True)

# DataLoaders: num_workers=2 works smoothly and safely with atomic caching
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"✔ DataLoaders configured: {len(train_loader)} train batches | {len(val_loader)} val batches.")

Preparing train and val datasets...
✔ DataLoaders configured: 691 train batches | 60 val batches.


In [6]:
# CELL 3: MAXVIT INITIALIZATION
import torch
import torch.nn as nn
import timm

print("=" * 70)
print("STEP 3: INITIALIZING GOOGLE MAXVIT BACKBONE")
print("=" * 70)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Device: {device}")
if device.type == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")

# Load pre-trained MaxViT Tiny (224x224 input)
model_arch = "maxvit_tiny_tf_224"
print(f"\nLoading '{model_arch}' with {num_classes} output variant classes...")
model = timm.create_model(model_arch, pretrained=True, num_classes=num_classes)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

# Training hyperparameters
num_epochs = 5
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

print("\n✔ Model, Criterion, Optimizer & Cosine Scheduler ready.")

STEP 3: INITIALIZING GOOGLE MAXVIT BACKBONE
Target Device: cuda
GPU Model: Tesla T4
Allocated VRAM: 122.1 MB

Loading 'maxvit_tiny_tf_224' with 1922 output variant classes...


Total Parameters:     31,389,514
Trainable Parameters: 31,389,514

✔ Model, Criterion, Optimizer & Cosine Scheduler ready.


In [ ]:
# CELL 4: TRAINING & VALIDATION LOOP WITH REAL-TIME CSV LOGGING
import os
import csv
import time

print("=" * 70)
print(f"STEP 4: STARTING FINE-TUNING LOOP ({num_epochs} EPOCHS)")
print("=" * 70)

# -------------------------------------------------------------------------
# 1. INITIALIZE CSV LOG FILES WITH HEADERS
# -------------------------------------------------------------------------
BATCH_CSV_FILE = "batch_metrics.csv"
EPOCH_CSV_FILE = "epoch_metrics.csv"

with open(BATCH_CSV_FILE, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch", 
        "batch", 
        "total_batches", 
        "batch_loss", 
        "running_acc", 
        "learning_rate", 
        "elapsed_sec"
    ])

with open(EPOCH_CSV_FILE, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch", 
        "train_loss", 
        "train_acc", 
        "val_loss", 
        "val_acc", 
        "learning_rate", 
        "duration_sec", 
        "is_best"
    ])

print(f"✔ Logging batch-level metrics to: '{BATCH_CSV_FILE}'")
print(f"✔ Logging epoch-level metrics to: '{EPOCH_CSV_FILE}'\n")

# -------------------------------------------------------------------------
# 2. TRAINING & VALIDATION LOOP
# -------------------------------------------------------------------------
best_val_acc = 0.0
total_train_batches = len(train_loader)

for epoch in range(1, num_epochs + 1):
    t0_epoch = time.time()
    lr_current = scheduler.get_last_lr()[0]
    print(f"\n>>> Epoch {epoch}/{num_epochs} [LR: {lr_current:.6f}]")

    # --- Training Phase ---
    model.train()
    running_loss, train_correct, train_total = 0.0, 0, 0

    for b_idx, (images, targets) in enumerate(train_loader, start=1):
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(targets)
        train_correct += (logits.argmax(dim=1) == targets).sum().item()
        train_total += len(targets)

        batch_acc = (train_correct / train_total) * 100
        cur_loss = running_loss / train_total
        elapsed_sec = time.time() - t0_epoch

        # Write batch row to CSV immediately
        with open(BATCH_CSV_FILE, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch,
                b_idx,
                total_train_batches,
                f"{cur_loss:.4f}",
                f"{batch_acc:.2f}",
                f"{lr_current:.6f}",
                f"{elapsed_sec:.2f}"
            ])

        # Print console progress every 100 batches or at end of epoch
        if b_idx % 100 == 0 or b_idx == total_train_batches:
            print(f"  [Train] Batch {b_idx:4d}/{total_train_batches} | Loss: {cur_loss:.4f} | Acc: {batch_acc:.2f}% | Elapsed: {elapsed_sec:.1f}s")

    scheduler.step()
    epoch_train_loss = running_loss / train_total
    epoch_train_acc = (train_correct / train_total) * 100

    # --- Validation Phase ---
    print(f"\n  [Val] Running validation on {len(val_dataset)} held-out store crops...")
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            logits = model(images)
            loss = criterion(logits, targets)

            val_loss += loss.item() * len(targets)
            val_correct += (logits.argmax(dim=1) == targets).sum().item()
            val_total += len(targets)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = (val_correct / val_total) * 100
    epoch_duration = time.time() - t0_epoch
    is_best = epoch_val_acc > best_val_acc

    # Write epoch row to CSV immediately
    with open(EPOCH_CSV_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch,
            f"{epoch_train_loss:.4f}",
            f"{epoch_train_acc:.2f}",
            f"{epoch_val_loss:.4f}",
            f"{epoch_val_acc:.2f}",
            f"{lr_current:.6f}",
            f"{epoch_duration:.2f}",
            is_best
        ])

    print(f"\n  Epoch {epoch} Summary ({epoch_duration:.1f}s):")
    print(f"    • Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
    print(f"    • Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.2f}%")

    # --- Save Best Model Checkpoint ---
    if is_best:
        improvement = epoch_val_acc - best_val_acc
        best_val_acc = epoch_val_acc
        print(f"    ★ New best validation accuracy (+{improvement:.2f}%)!")

        # 1. Save PyTorch state dictionary
        torch.save(model.state_dict(), "best_maxvit_brand_variant.pth")

        # 2. Export TorchScript model
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224, device=device)
            traced = torch.jit.trace(model, dummy_input)
            traced.save("traced_maxvit_brand_variant.pt")
        print("    ✔ Exported 'best_maxvit_brand_variant.pth' and 'traced_maxvit_brand_variant.pt'")

print("\n" + "=" * 70)
print(f"TRAINING COMPLETE! Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Metrics saved to '{BATCH_CSV_FILE}' and '{EPOCH_CSV_FILE}'.")
print("=" * 70)

STEP 4: STARTING FINE-TUNING LOOP (5 EPOCHS)
✔ Logging batch-level metrics to: 'batch_metrics.csv'
✔ Logging epoch-level metrics to: 'epoch_metrics.csv'


>>> Epoch 1/5 [LR: 0.000100]
  [Train] Batch  100/691 | Loss: 7.4114 | Acc: 2.84% | Elapsed: 300.5s
  [Train] Batch  200/691 | Loss: 7.1798 | Acc: 7.59% | Elapsed: 577.0s
  [Train] Batch  300/691 | Loss: 6.9507 | Acc: 10.98% | Elapsed: 860.2s
  [Train] Batch  600/691 | Loss: 6.2272 | Acc: 19.94% | Elapsed: 1679.6s
  [Train] Batch  300/691 | Loss: 2.1305 | Acc: 88.22% | Elapsed: 224.7s
  [Train] Batch  400/691 | Loss: 2.1207 | Acc: 88.05% | Elapsed: 299.4s
  [Train] Batch  500/691 | Loss: 2.1105 | Acc: 88.21% | Elapsed: 374.1s
  [Train] Batch  600/691 | Loss: 2.0948 | Acc: 88.29% | Elapsed: 448.8s
  [Train] Batch  691/691 | Loss: 2.0832 | Acc: 88.52% | Elapsed: 516.9s

  [Val] Running validation on 1900 held-out store crops...

  Epoch 4 Summary (531.4s):
    • Train Loss: 2.0832 | Train Acc: 88.52%
    • Val Loss:   2.6537 | Val Acc: